In [1]:
texto = """
A Microsoft introduziu um ambiente operacional chamado Windows 1.0 em 20 de novembro de 1985, como um shell para MS-DOS, 
em resposta ao crescente interesse em interfaces gráficas de utilizador (GUIs).[1]
O Microsoft Windows passou a dominar o mercado de computadores pessoais (PC) do mundo, 
com mais de 90% de participação de mercado, superando o macOS, que havia sido introduzido em 1984. 
A Apple chegou a ver o Windows como uma invasão injusta em sua inovação no desenvolvimento de produtos GUI, 
como o Lisa e o Macintosh (eventualmente resolvido na Justiça em favor da Microsoft em 1993). 
Nos PCs, o Windows ainda é o sistema operacional mais popular do mundo.
"""

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List
from langchain_core.utils.function_calling import convert_to_openai_function

In [3]:
# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key = os.getenv("ROUTER_API"),
    model="gpt-oss-120b:free", 
    temperature=0.7
)

In [11]:
from typing import List
from pydantic.v1 import BaseModel, Field

class Event(BaseModel):
    """Informações sobre um evento ocorrido"""
    
    date: str = Field(
        description="Data do evento no formato YYYY-MM-DD"
    )
    
    event: str = Field(
        description="Descrição do evento extraído do texto"
    )

class EventsList(BaseModel):
    """Lista de eventos"""
    
    events: List[Event] = Field(
        description="Conjunto de eventos encontrados"
    )


In [12]:
from langchain_core.output_parsers.openai_tools import PydanticToolsParser

In [13]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extraia as frases de acontecimentos e as extraia integralmente"),
    ("user", "{input}")
])


parse = PydanticToolsParser(
    tools=[EventsList]
)

chain = prompt | chat.bind_tools([EventsList]) | parse

resposta = chain.invoke({"input": texto})
# print(resposta.tool_calls[0]['args'])
print(resposta)

[EventsList(events=[Event(date='1985-11-20', event='Microsoft introduziu um ambiente operacional chamado Windows 1.0 como um shell para MS-DOS.'), Event(date='1993-01-01', event='A Justiça decidiu a favor da Microsoft no caso envolvendo a alegada invasão injusta da Apple.')])]


### Dados da Web (incompleto)

In [ ]:
from langchain_community.document_loaders.web_base import WebBaseLoader

loader = WebBaseLoader("https://www.techtudo.com.br")
page = loader.load()
page

In [ ]:
class BlogPost(BaseModel):
    '''Detalhes sobre uma postagem de blog'''
    title: str = Field(description="Titulo da postagem no blog")
    author: str = Field(description="Nome do autor da postagem")

class BlogSite(BaseModel):
    """Conjunto de postagens de blog e um site específico"""
    posts: List[BlogPost] = Field(description="Coleção de postagens e blog do site")

tool_blog = convert_to_openai_function(BlogSite)
tool_blog

{'name': 'BlogSite',
 'description': 'Conjunto de postagens de blog e um site específico',
 'parameters': {'properties': {'posts': {'description': 'Coleção de postagens e blog do site',
    'items': {'description': 'Detalhes sobre uma postagem de blog',
     'properties': {'title': {'description': 'Titulo da postagem no blog',
       'type': 'string'},
      'author': {'description': 'Nome do autor da postagem',
       'type': 'string'}},
     'required': ['title', 'author'],
     'type': 'object'},
    'type': 'array'}},
  'required': ['posts'],
  'type': 'object'}}

In [ ]:
from langchain_core.output_parsers.openai_tools import PydanticToolsParser

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extraia todas as postagens do blog presentes na página. Para cada postagem, identifique título e autor."),
    ("user", "{input}")
])

parse = PydanticToolsParser(tools=[BlogSite])

chain = prompt | chat.bind_tools([tool_blog], tool_choice="auto") | parse

content = page[0].page_content
chain.invoke({"input": content})

[]